# 08c. Variance attribution on D2 and D3

Extension of the D1 factorial (notebook 03). Models: GBDT, RF, logistic regression, MLP.

- D3: 4 models × 4 policies (S1 segment-random, S2b run-independent, S3 LOCO, S3′ LOTO) × 10 seeds; OOF over 3,600 segments (fold seed = model seed)
- D2: 4 models × 2 protocols (naive 70/30, logger-independent 65/35) × 10 seeds; 140 WAV, 2023 mel features
- Shares are descriptive for these models and protocols only

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd
import src as la
from src import config as cfg, io_utils, evaluation as ev, sfb, reporting as rp
cfg.apply_mpl_style(); cfg.banner()
from sklearn.model_selection import train_test_split, GroupShuffleSplit
import warnings; warnings.filterwarnings("ignore")

LeakAudit | root=C:\Users\ecredible\Desktop\LeakAudit_revision1\LeakAudit | mode=FULL | seeds=10 | bootstrap=2000


## 1. D3

In [3]:
M3 = io_utils.scan_d3()
H = M3[(~M3.is_bg) & (M3.modality == "hydrophone") & M3.label.notna()].reset_index(drop=True)
c = io_utils.load_cache(f"d3_seg_hydrophone_{cfg.WIN_SEC}_pcm32", H.file_id, by_unique=True)
assert c is not None, "run notebook 06 first"
S = H.set_index("file_id").loc[c["ids"].astype(str)].reset_index()
X, y = c["X"], S.label.to_numpy().astype(str)
pol = {"S1_segment_random": ("stratified", None), "S2b_run_indep": ("stratified_group", S.run_id.to_numpy()),
       "S3_opcond_LOCO": ("logo", S.op_condition.to_numpy()), "S3_topology_LOTO": ("logo", S.topology.to_numpy())}
rows = []
for kind in ev.MODELS:
    for seed in range(cfg.N_SEEDS):
        for p, (scheme, g) in pol.items():
            folds = ev.make_folds(scheme, y, g, 5, seed=seed)
            pred, _ = ev.oof_predict(X, y, folds, kind, seed)
            s = ev.score(y, pred)
            rows.append(dict(dataset="D3", model=kind, protocol=p, seed=seed, acc=s["acc"], bacc=s["bacc"]))
    print(f"  D3 {kind} done", flush=True)
f3 = pd.DataFrame(rows)

  D3 gbdt done
  D3 rf done
  D3 logreg done
  D3 mlp done


## 2. D2

Draws without both classes in train or test skipped

In [4]:
M2 = io_utils.scan_d2()
W = M2[(M2.ext == ".wav") & M2.label.isin(["leak", "noleak"])].reset_index(drop=True)
cm = io_utils.load_cache("d2_emb_Mel2023", W.file_id)
assert cm is not None, "run notebook 07 first"
X2, y2, lg = cm["X"], W.label.to_numpy(), W.logger.fillna("NA").to_numpy()
rows = []
for kind in ev.MODELS:
    for seed in range(cfg.N_SEEDS):
        for p in ["P1_naive", "P2_logger_indep"]:
            if p == "P1_naive":
                a, b = train_test_split(np.arange(len(y2)), test_size=.3, random_state=seed, stratify=y2)
            else:
                a, b = next(GroupShuffleSplit(1, test_size=.35, random_state=seed).split(X2, y2, lg))
            if len(set(y2[a])) < 2 or len(set(y2[b])) < 2:
                continue
            r = ev.evaluate(X2[a], y2[a], X2[b], y2[b], kind, seed)
            rows.append(dict(dataset="D2", model=kind, protocol=p, seed=seed, acc=r["acc"], bacc=r["bacc"]))
f2 = pd.DataFrame(rows)

## 3. Variance attribution by benchmark (accuracy, BA)

D1 rows from notebook 03 (accuracy only); output: `results/E4x_variance_attribution_all.csv`

In [5]:
fx = pd.concat([f2, f3], ignore_index=True)
fx.to_csv(cfg.RESULTS / "E4x_factorial_D2_D3_raw.csv", index=False)
tabs, cells = [], []
for ds, g in fx.groupby("dataset"):
    for metric in ["acc", "bacc"]:
        tab, cm_ = ev.factorial_ss(g, "protocol", "model", metric)
        tab.insert(0, "metric", metric); tab.insert(0, "dataset", ds)
        cm_.insert(0, "metric", metric); cm_.insert(0, "dataset", ds)
        tabs.append(tab); cells.append(cm_)
va = pd.concat(tabs, ignore_index=True)
pd.concat(cells, ignore_index=True).to_csv(cfg.RESULTS / "E4x_cell_means.csv", index=False)
p1 = cfg.RESULTS / "E4_variance_attribution.csv"
if p1.exists():
    d1 = pd.read_csv(p1).assign(metric="acc")
    d1 = d1.rename(columns={"task": "dataset"})
    d1["dataset"] = "D1:" + d1["dataset"]
    va = pd.concat([d1[va.columns.intersection(d1.columns)], va], ignore_index=True)
va.to_csv(cfg.RESULTS / "E4x_variance_attribution_all.csv", index=False)
print(va[["dataset", "metric", "term", "ss_share", "partial_eta2", "n_fits", "balanced"]].round(4).to_string(index=False))
for _, r in va[va.dataset.isin(["D2", "D3"])].iterrows():
    rp.record(f"{r.dataset}.factorial.{r.metric}.{r.term}.share", r.ss_share,
              "E4x_variance_attribution_all.csv", "R1-M6,R2-9", note=f"n_fits={r.n_fits}")

              dataset metric             term  ss_share  partial_eta2  n_fits  balanced
     D1:T5_five_class    acc         protocol    0.8095        0.8761     160      True
     D1:T5_five_class    acc            model    0.0254        0.1816     160      True
     D1:T5_five_class    acc protocol x model    0.0507        0.3069     160      True
     D1:T5_five_class    acc  residual (seed)    0.1144           NaN     160      True
D1:TL_leak_vs_nonleak    acc         protocol    0.7913        0.8478     160      True
D1:TL_leak_vs_nonleak    acc            model    0.0252        0.1505     160      True
D1:TL_leak_vs_nonleak    acc protocol x model    0.0414        0.2258     160      True
D1:TL_leak_vs_nonleak    acc  residual (seed)    0.1421           NaN     160      True
      D1:TR_out_vs_in    acc         protocol    0.8414        0.9563     160      True
      D1:TR_out_vs_in    acc            model    0.0258        0.4019     160      True
      D1:TR_out_vs_in    acc pro